# **PetVision_v2**

Vet clinic internal assistant agentic chatbot for administrator to automate appointment process.

# **Load credentials.json From Google Drive**

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

CREDENTIALS_PATH = "/content/drive/MyDrive/PetVision_v2/credentials.json"

# **Installation and Setup**

In [ ]:
# Install google auth
!pip install --upgrade google-api-python-client google-auth-httplib2 google-auth-oauthlib

# Install langchain community
!pip install -q langchain-community

# Import secret dependecies
from google.colab import userdata

In [ ]:
# Set up Google Calendar API with Colab credentials
from googleapiclient.discovery import build
from google_auth_oauthlib.flow import Flow

SCOPES = ["https://www.googleapis.com/auth/calendar"]
TOKEN_PATH = "token.json"

creds = None

flow = Flow.from_client_secrets_file(
    CREDENTIALS_PATH,
    scopes=SCOPES,
    redirect_uri="urn:ietf:wg:oauth:2.0:oob"  # OOB for manual copy-paste
)

auth_url, _ = flow.authorization_url(prompt="consent")
print("Visit this URL to authorize:\n", auth_url)

code = input("\nPaste the authorization code here: ")
flow.fetch_token(code=code)
creds = flow.credentials

with open(TOKEN_PATH, "w") as token:
    token.write(creds.to_json())

service = build("calendar", "v3", credentials=creds)

In [ ]:
# Setup PostgreSQL
from sqlalchemy import create_engine, text
from google.colab import userdata

DB_URL = userdata.get("SUPABASE_DB_URL")
engine = create_engine(DB_URL)

# TODO: Dummy ID for testing only
VISIT_ID = "0863b510-c050-484f-8d13-700537cee5e2"
CLINIC_LOCATION = "PetVision Clinic Penang"

In [ ]:
# Setup chat model
# Qwen2.5-1.5B: https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from langchain_community.llms import HuggingFacePipeline
import torch

model_id = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_id)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="auto"
)

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=512,
    temperature=0.1,
    do_sample=True,
    return_full_text=False
)

llm = HuggingFacePipeline(pipeline=pipe)

# **Create an Event**

In [ ]:
# Insert event into PostgreSQL
from sqlalchemy import text

def insert_event(visit_id: str, calendar_event_id: str):
  query = text("""
      UPDATE visits
      SET calendar_event_id = :calendar_event_id
      WHERE visit_id = :visit_id;
  """)

  # Execute the query
  # Open a database transaction connection
  with engine.begin() as connection:

    # Execute the SQL query with parameter values
    connection.execute(
      query,
      {"visit_id": visit_id, "calendar_event_id": calendar_event_id}
    )

  print("Event ID inserted successfully.")

In [ ]:
# Create Google Calendar event and save event ID into PostgreSQL
from googleapiclient.errors import HttpError

def create_event(service, visit_id, event_body):

  # Initialize variable for storing Google Calendar event ID
  calendar_event_id = None

  try:
    # Create event in Google Calendar
    gcal_event = service.events().insert(
        calendarId="primary",
        body=event_body
    ).execute()

    # Extract Google Calendar event ID from API response
    calendar_event_id = gcal_event["id"]

  except HttpError as gcal_error:
    return {
      "success": False,
      "stage": "google_calendar",
      "error": f"Google Calendar API Error: {gcal_error}"
    }

  except Exception as gcal_error:
    return {
      "success": False,
      "stage": "google_calendar",
      "error": str(gcal_error)
    }

  try:
    # Save Google Calendar event ID into PostgreSQL
    insert_event(
      visit_id=visit_id,
      calendar_event_id=calendar_event_id
    )

    # Return success response after both operations succeed
    return {
      "success": True,
      "message": "Event created and saved successfully",
      "visit_id": visit_id,
      "calendar_event_id": calendar_event_id,
      "html_link": gcal_event.get("htmlLink"),
      "event": gcal_event
    }

  except Exception as db_error:

    # Attempt rollback by deleting Google Calendar event
    try:
      service.events().delete(
          calendarId="primary",
          eventId=calendar_event_id
      ).execute()

      rollback_status = "Google event rolled back successfully"

    except Exception as rollback_error:
      rollback_status = f"Rollback failed: {rollback_error}"

    return {
      "success": False,
      "stage": "database",
      "error": str(db_error),
      "rollback_status": rollback_status
    }

# **Read Events**

In [ ]:
# Retrieve Google Calendar event ID from PostgreSQL
from sqlalchemy import text

def get_event_id(visit_id: str) -> str:

  query = text("""
      SELECT v.calendar_event_id
      FROM visits v
      WHERE visit_id = :visit_id;
  """)

  # Execute the query
  # Open database transaction connection
  with engine.begin() as connection:

    # Execute SQL query with visit_id parameter and fetch one result
    result = connection.execute(
        query,
        {"visit_id": visit_id}
    ).fetchone()

    # Return None if no matching record is found
    if result is None:
      return None

     # Return the calendar_event_id column value
    return result[0]

In [ ]:
# Retrieve Google Calendar event details
from googleapiclient.errors import HttpError

def get_event(service, visit_id):
  try:
    # Retrieve Google Calendar event ID from PostgreSQL
    event_id = get_event_id(visit_id)

    # Return error response if event ID is missing
    if not event_id:
      return {
        "success": False,
        "error": "Event ID not found"
      }

    # Fetch event from Google Calendar
    event = service.events().get(
      calendarId="primary",
      eventId=event_id
    ).execute()

    # Extract details
    event_details = {
      "success": True,
      "event_id": event_id,
      "title": event.get("summary", "No Title"),
      "start": event["start"].get("dateTime", event["start"].get("date")),
      "end": event["end"].get("dateTime", event["end"].get("date")),
      "location": event.get("location", "No location"),
      "description": event.get("description", "No description"),
      "attendees": event.get("attendees", [])
    }

    return event_details

  # Handle Google Calendar API-specific errors
  except HttpError as error:
    return {
      "success": False,
      "error": f"Google Calendar API Error: {error}"
    }
  # Handle unexpected runtime errors
  except Exception as e:
    return {
      "success": False,
      "error": str(e)
    }

# **Update an Existing Event**

In [ ]:
# Update an existing Google Calendar event
from googleapiclient.errors import HttpError

def update_event(service, visit_id, updated_fields):
  try:
    # Retrieve Google Calendar event ID from PostgreSQL
    event_id = get_event_id(visit_id)

    # Return error response if event ID is missing
    if not event_id:
      return {
        "success": False,
        "error": "Event ID not found"
      }

    # Update Google Calendar event
    event = service.events().patch(
      calendarId="primary",
      eventId=event_id,
      body=updated_fields
    ).execute()

    # Sleep for 5 seconds to avoid synchronization issue
    sleep(5)

    # Return successful update response
    return {
      "success": True,
      "message": "Event updated successfully",
      "event_id": event_id,
      "html_link": event.get("htmlLink"),
      "updated_event": event
    }

  except HttpError as error:
      return {
        "success": False,
        "error": f"Google Calendar API Error: {error}"
      }

  except Exception as e:
      return {
        "success": False,
        "error": str(e)
      }

# **Delete an Existing Event**

In [ ]:
# Delete calendar_event_id reference from PostgreSQL
from sqlalchemy import text

def remove_calendar_event_id(visit_id: str):
  query = text("""
      UPDATE visits
      SET calendar_event_id = NULL
      WHERE visit_id = :visit_id
  """)

  # Open database transaction connection
  with engine.begin() as connection:

    # Execute SQL query with visit_id parameter
    connection.execute(query, {"visit_id": visit_id})

  print("Calendar event ID removed from DB")

In [ ]:
# Delete Google Calendar event and clean up DB reference
from googleapiclient.errors import HttpError

def delete_event(service, visit_id: str):

  try:
    # Retrieve Google Calendar event ID from PostgreSQL
    calendar_event_id = get_event_id(visit_id)

    # Return error if no linked event exists
    if not calendar_event_id:
      return {
        "success": False,
        "error": "No calendar event linked"
      }

    # Delete event from Google Calendar
    service.events().delete(
      calendarId="primary",
      eventId=calendar_event_id
    ).execute()

  # Handle Google Calendar API-specific errors
  except HttpError as gcal_error:
    return {
      "success": False,
      "stage": "google_calendar",
      "error": f"Google Calendar API Error: {gcal_error}"
    }

  # Handle unexpected errors during Google Calendar deletion
  except Exception as gcal_error:
    return {
      "success": False,
      "stage": "google_calendar",
      "error": str(gcal_error)
    }

  try:
    # Remove calendar event reference from PostgreSQL
    remove_calendar_event_id(visit_id)

    # Return success response after full cleanup
    return {
      "success": True,
      "message": "Event deleted successfully",
      "visit_id": visit_id,
      "calendar_event_id": calendar_event_id
    }

  except Exception as db_error:
    return {
      "success": False,
      "stage": "database",
      "error": str(db_error),
      "warning": "Google event deleted but DB cleanup failed"
    }

# **Qwen2.5 Generation**

In [ ]:
# Define tools as JSON schemas
tools = [
  {
    "type": "function",
    "function": {
        "name": "create_event",
        "description": "Create a new vet appointment in Google Calendar and link it to a visit in the DB",
        "parameters": {
            "type": "object",
            "properties": {
                "visit_id":    {"type": "string", "description": "UUID of the visit record"},
                "summary":     {"type": "string", "description": "Appointment title"},
                "start_time":  {"type": "string", "description": "ISO 8601 datetime, e.g. 2026-06-01T10:00:00+08:00"},
                "end_time":    {"type": "string", "description": "ISO 8601 datetime"},
                "description": {"type": "string"},
                "attendees": {
                    "type": "array",
                    "items": {"type": "object", "properties": {"email": {"type": "string"}}},
                    "description": "List of attendees, e.g. [{\"email\": \"dr.tan@clinic.com\"}]"
                },
                "location": {"type": "string"},
            },
            "required": ["visit_id", "summary", "start_time", "end_time"]
        }
    }
  },
  {
    "type": "function",
    "function": {
        "name": "get_event",
        "description": "Fetch appointment details for a visit",
        "parameters": {
            "type": "object",
            "properties": {
                "visit_id": {"type": "string"}
            },
            "required": ["visit_id"]
        }
    }
  },
  {
    "type": "function",
    "function": {
        "name": "update_event",
        "description": "Update fields of an existing appointment",
        "parameters": {
            "type": "object",
            "properties": {
                "visit_id": {"type": "string"},
                "fields":   {"type": "object", "description": "Key-value pairs to update, e.g. {summary, location}"}
            },
            "required": ["visit_id", "fields"]
        }
    }
  },
  {
    "type": "function",
    "function": {
        "name": "delete_event",
        "description": "Delete a vet appointment from Google Calendar and clear the DB reference",
        "parameters": {
            "type": "object",
            "properties": {
                "visit_id": {"type": "string"}
            },
            "required": ["visit_id"]
        }
    }
  },
]

In [ ]:
# Central dispatcher to route tool calls to Google Calendar and DB functions
import json

def dispatch_tool(tool_name: str, args: dict):

  # Handle event creation
  if tool_name == "create_event":

    # Build event body
    event_body = {
      "summary": args["summary"],
      "start": {
          "dateTime": args["start_time"],
          "timeZone": "Asia/Kuala_Lumpur"
      },
      "end": {
          "dateTime": args["end_time"],
          "timeZone": "Asia/Kuala_Lumpur"
      },
      "description": args.get("description", ""),
      "location": args.get("location", "")
    }

    # Add attendees only if provided
    if args.get("attendees"):
      event_body["attendees"] = args["attendees"]

    return create_event(
      service,
      args["visit_id"],
      event_body
    )

  # Handle fetching an existing event
  elif tool_name == "get_event":
    return get_event(service, args["visit_id"])

  # Handle updating an existing event
  elif tool_name == "update_event":
    return update_event(
      service,
      args["visit_id"],
      args["fields"]
    )

  # Handle deleting an event
  elif tool_name == "delete_event":
    delete_event(service, args["visit_id"])
    return {"success": True, "message": "Event deleted successfully"}

  # Return error for unsupported tool name
  else:
    return {
      "success": False,
      "error": f"Unknown tool: {tool_name}"
    }

In [ ]:
import re

# Extract tool call content wrapped inside <tool_call> tags
def extract_tool_call(text: str) -> str:

  # Regex search for content inside tool_call XML-like tags
  match = re.search(r"<tool_call>(.*?)</tool_call>", text, re.DOTALL)

  # If a match is found, return the extracted tool call string
  if match:
    return match.group(1).strip()

  # Raise error if no tool_call block is found in the input text
  raise ValueError("No tool_call found in model response")

In [ ]:
# Handles LLM responses and optional tool execution
def chat(user_message: str, history: list = None):

  SYSTEM_PROMPT = f"""You are a vet clinic assistant.
  The current visit ID is {VISIT_ID}.
  The location of the clinic is {CLINIC_LOCATION}.
  Use it for all appointment operations."""

  # Initialize history list
  if history is None:
    history = []

  # Append new user message to conversation history
  messages = [{"role": "system", "content": SYSTEM_PROMPT}] + history + [{"role": "user", "content": user_message}]

  # Convert conversation into model-ready prompt using chat template
  prompt = tokenizer.apply_chat_template(
    messages,
    tools=tools,
    add_generation_prompt=True,
    tokenize=False
  )

  # Run first inference pass through the model
  output = pipe(prompt)

  # Extract generated assistant response
  assistant_msg = output[0]["generated_text"]

  if "<tool_call>" in assistant_msg:

    # Extract JSON content inside tool_call tags
    tool_call = json.loads(extract_tool_call(assistant_msg))

    # Execute requested tool with provided arguments
    tool_result = dispatch_tool(
      tool_call["name"],
      tool_call["arguments"]
    )

    # Append assistant tool-call message to conversation history
    messages.append(
        {
          "role": "assistant",
          "content": assistant_msg,
        }
    )

    # Append tool execution result to conversation history
    messages.append(
        {
          "role": "tool",
          "content": json.dumps(tool_result),
          "tool_call_id": tool_call.get("id", "0"),
        }
    )

    # Second LLM call to get natural-language reply
    second_prompt = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=False
    )

    # Run second inference pass to generate final natural-language response
    final = pipe(second_prompt)

    # Return final assistant response
    return final[0]["generated_text"]

  return assistant_msg

# **RAG Pipeline**

In [ ]:
history = []

# 1. CREATE
query_1 = """Book an appointment at 29/5/2026, 2pm to 3pm.
    Title: General Checkup.
    Attendee email: xxx@gmail.com"""
conversation_1 = chat(query_1, history)

print(conversation_1)

history += [
    {"role": "user", "content": query_1},
    {"role": "assistant", "content": conversation_1}
]

print(history)

# 2. READ
query_2 = "Thanks! Can you remind me what time the appointment is?"
conversation_2 = chat(query_2, history)

print(conversation_2)

history += [
    {"role": "user", "content": query_2},
    {"role": "assistant", "content": conversation_2}
]

print(history)

# 3. UPDATE
query_3 = "Actually, move it to 4pm to 5pm"

conversation_3 = chat(query_3, history)

print(conversation_3)

history += [
    {"role": "user", "content": query_3},
    {"role": "assistant", "content": conversation_3}
]

print(history)

# 4. DELETE
query_4 = "Please cancel the appointment for me"

conversation_4 = chat(query_4, history)

print(conversation_4)

history += [
    {"role": "user", "content": query_4},
    {"role": "assistant", "content": conversation_4}
]

print(history)